# 02 Modeling: P(success | age, Ivy League, degree level, batch)

Run this top to bottom yourself. Nothing here has been run on your real data.

**What each section answers (your professor's list)**
1. P(success | founder age)
2. P(success | went to an Ivy League school)
3. Do people with a master's (vs. bachelor's only) succeed more? (I read "batch or masters" as "bachelor's or master's". Section 4 also covers YC batch in case that is what was meant.)

**How success is defined (change in the config cell if the professor wants something else)**
- success = `acquired` or `public`; failure = `inactive`
- `active` and `unclear` companies are left out, because their outcome is not decided yet (censored)

**Put `founders_linkedin_filled_final.xlsx` in the same folder as this notebook.**

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from scipy import stats

DATA_PATH = "founders_linkedin_filled_final.xlsx"
ASSUMED_GRAD_AGE = 22                      # same assumption as your sheet (cell S1)
SUCCESS_STATUSES = {"acquired", "public"}
FAILURE_STATUSES = {"inactive"}            # try adding "active" to SUCCESS_STATUSES for a sensitivity check

BLUE, ORANGE, GRAY, INK, INK2 = "#2a78d6", "#eb6834", "#b9b8b2", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titlesize": 12, "axes.titleweight": "bold",
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
})

## 0. Load and build features

In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name=0)
raw.columns = [str(c).strip() for c in raw.columns]

df = raw.rename(columns={
    "fullName": "founder", "Verified Founding Year": "founding_year_verified",
    "Bachelor's Grad Year": "grad_year", "Published Age": "published_age",
    "Year Age Published": "year_age_published", "Basis": "basis", "Age Method": "age_method",
})
print(df.shape)
df.head(3)

In [ ]:
# --- Age at founding (mirrors the formula in your sheet's column N) ---
age_published = df["published_age"] + (df["founding_year_verified"] - df["year_age_published"])
age_from_grad = df["founding_year_verified"] - df["grad_year"] + ASSUMED_GRAD_AGE
df["age"] = age_published.fillna(age_from_grad)

# --- Outcome ---
df["success"] = np.select(
    [df["current_status"].isin(SUCCESS_STATUSES), df["current_status"].isin(FAILURE_STATUSES)],
    [1.0, 0.0], default=np.nan)

# --- Education flags, parsed from the free-text 'basis' column ---
basis = df["basis"].fillna("").astype(str)

IVY_RE      = re.compile(r"\b(Harvard|HBS|Yale|Princeton|Columbia|Cornell|Dartmouth|Brown(?: Univ(?:ersity)?| \()|UPenn|U Penn|Penn(?! State)|Wharton|University of Pennsylvania)\b")
NOT_IVY_RE  = re.compile(r"British Columbia|Columbia Basin|Columbia College Chicago|Columbia Southern|Penn State|Pennsylvania State|Penn Almanac|deferred \w+|Fulbright at \w+")
MASTERS_RE  = re.compile(r"[Mm]aster|\bMBA\b|\bMSc\b|\bMEng\b|\bMPhil\b|\bMFA\b|\bMPA\b|\bMPP\b|\bMS\b|\bM\.S\.")
PHD_RE      = re.compile(r"Ph\.?D|[Dd]octorate|\bDPhil\b")
DROPOUT_RE  = re.compile(r"[Dd]ropout|[Dd]ropped out|[Dd]id not graduate")
NO_EDU_RE   = re.compile(r"^\s*[Nn]o\b[^;]*\beducation")      # e.g. 'No education dates found'

df["edu_known"] = basis.ne("") & ~basis.map(lambda s: bool(NO_EDU_RE.search(s)))
df["ivy"]     = basis.map(lambda s: bool(IVY_RE.search(NOT_IVY_RE.sub("", s))))
df["masters"] = basis.map(lambda s: bool(MASTERS_RE.search(s)))
df["phd"]     = basis.map(lambda s: bool(PHD_RE.search(s)))
df["dropout"] = basis.map(lambda s: bool(DROPOUT_RE.search(s)))

# Manual fixes after you audit the matches below: put row numbers (the DataFrame index) here.
IVY_FORCE_TRUE, IVY_FORCE_FALSE = [], []
df.loc[IVY_FORCE_TRUE, "ivy"] = True
df.loc[IVY_FORCE_FALSE, "ivy"] = False

# Highest degree found (only defined where we actually know their education)
df["degree_level"] = np.select(
    [~df["edu_known"], df["phd"], df["masters"], df["dropout"]],
    [None, "phd", "masters", "dropout"], default="bachelors_only")
df.loc[~df["edu_known"], "degree_level"] = np.nan

# --- Controls ---
df["founding_year_c"] = df["founding_year_verified"] - df["founding_year_verified"].mean()
top_ind = df["industry"].astype(str).str.strip().value_counts().head(8).index
df["industry_grp"] = np.where(df["industry"].astype(str).str.strip().isin(top_ind),
                              df["industry"].astype(str).str.strip(), "Other")

# --- YC batch -> season + year ---
parts = df["batch"].astype(str).str.extract(r"^([A-Za-z]+)\s*(\d{2,4})$")
df["batch_season"] = parts[0]
yr = pd.to_numeric(parts[1], errors="coerce")
df["batch_year"] = np.where(yr < 100, 2000 + yr, yr)

In [ ]:
coverage = pd.Series({
    "founder rows": len(df),
    "with a known outcome (success/failure)": df["success"].notna().sum(),
    "with a computed age": df["age"].notna().sum(),
    "with a computed age AND known outcome": (df["age"].notna() & df["success"].notna()).sum(),
    "with education info in 'basis'": df["edu_known"].sum(),
    "with education info AND known outcome": (df["edu_known"] & df["success"].notna()).sum(),
    "flagged Ivy League": df["ivy"].sum(),
    "flagged master's": df["masters"].sum(),
    "flagged PhD": df["phd"].sum(),
}, name="rows")
coverage.to_frame()

### Audit the education flags first
The flags come from keyword matching on free text, so check them before trusting any result. Look for false positives (e.g. a school mentioned but never attended) and add those row numbers to `IVY_FORCE_FALSE` in the cell above, then re-run from there.

In [ ]:
pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 200)
print("IVY matches:");      display(df.loc[df["ivy"], ["founder", "basis"]])

In [ ]:
print("MASTER'S matches:"); display(df.loc[df["masters"], ["founder", "basis"]])

In [ ]:
print("Random sample of rows with education info but NO flags (spot check for misses):")
display(df.loc[df["edu_known"] & ~df["ivy"] & ~df["masters"] & ~df["phd"], ["founder", "basis"]].sample(25, random_state=0))

## Helpers
`rate_table` gives P(success) per group with a 95% Wilson confidence interval. With small groups the intervals will be wide, which is itself a finding to report.

In [ ]:
def rate_table(data, group_col, order=None):
    d = data.dropna(subset=["success", group_col])
    g = d.groupby(group_col, observed=True)["success"].agg(successes="sum", n="count")
    lo, hi = proportion_confint(g["successes"], g["n"], method="wilson")
    g["p_success"], g["ci_low"], g["ci_high"] = g["successes"] / g["n"], lo, hi
    return g.reindex(order) if order is not None else g

def plot_rates(tbl, title, xlabel, baseline=None, colors=None, labels=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(tbl))
    cols = colors or [BLUE] * len(tbl)
    ax.bar(x, tbl["p_success"], width=0.55, color=cols)
    ax.errorbar(x, tbl["p_success"],
                yerr=[tbl["p_success"] - tbl["ci_low"], tbl["ci_high"] - tbl["p_success"]],
                fmt="none", ecolor=INK2, elinewidth=1.2, capsize=4)
    for xi, (p, n, hi) in enumerate(zip(tbl["p_success"], tbl["n"], tbl["ci_high"])):
        ax.text(xi, hi + 0.02, f"{p:.0%}\nn={int(n)}", ha="center", va="bottom", fontsize=9, color=INK)
    if baseline is not None:
        ax.axhline(baseline, color=GRAY, ls="--", lw=1.2)
        ax.text(len(tbl) - 0.5, baseline + 0.012, f"overall {baseline:.0%}", ha="right", va="bottom", fontsize=9, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels(labels or [str(i) for i in tbl.index])
    ax.set_ylim(0, 1.15); ax.set_ylabel("P(success)  (acquired or public)")
    ax.set_xlabel(xlabel); ax.set_title(title)
    plt.tight_layout(); plt.show()

def fit_glm(formula, data):
    # Logistic regression, standard errors clustered by company (founders at the same company are not independent)
    d = data.copy()
    groups = pd.factorize(d["company_name"])[0]
    return smf.glm(formula, data=d, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": groups})

def odds_ratios(res):
    ci = res.conf_int()
    out = pd.DataFrame({"odds_ratio": np.exp(res.params), "ci_low": np.exp(ci[0]),
                        "ci_high": np.exp(ci[1]), "p_value": res.pvalues})
    return out.round(3)

overall = df["success"].mean()
print(f"Overall success rate among resolved founder rows: {overall:.1%}")

## 1. P(success | age at founding)

**1a. Descriptive:** success rate by age band. Edit `AGE_BINS` if a band is too thin.

In [ ]:
AGE_BINS   = [0, 22, 25, 28, 31, 35, 100]
AGE_LABELS = ["22 or under", "23-25", "26-28", "29-31", "32-35", "36+"]
age_df = df.dropna(subset=["age", "success"]).copy()
age_df["age_band"] = pd.cut(age_df["age"], bins=AGE_BINS, labels=AGE_LABELS)

age_tbl = rate_table(age_df, "age_band", order=AGE_LABELS)
display(age_tbl.round(3))
plot_rates(age_tbl, "Success rate by founder age at founding", "Age at founding",
           baseline=age_df["success"].mean())

**1b. Model:** logistic regression with age and age squared (lets the effect rise then fall), then the same model with controls. Age is centered so the two terms are less correlated.

In [ ]:
m_df = df.dropna(subset=["age", "success", "founding_year_c"]).copy()
m_df["age_c"] = m_df["age"] - m_df["age"].mean()

m1 = fit_glm("success ~ age_c + I(age_c**2)", m_df)
print(m1.summary())
display(odds_ratios(m1))

In [ ]:
m2 = fit_glm("success ~ age_c + I(age_c**2) + founding_year_c + C(industry_grp)", m_df)
print(m2.summary())
display(odds_ratios(m2))

**1c. Predicted P(success) by age from model 1, with a 95% confidence band.**

In [ ]:
grid_ages = np.linspace(m_df["age"].quantile(0.05), m_df["age"].quantile(0.95), 80)
grid = pd.DataFrame({"age_c": grid_ages - m_df["age"].mean()})
pred = m1.get_prediction(grid).summary_frame()

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(grid_ages, pred["mean_ci_lower"], pred["mean_ci_upper"], color=BLUE, alpha=0.15, lw=0)
ax.plot(grid_ages, pred["mean"], color=BLUE, lw=2)
ax.axhline(m_df["success"].mean(), color=GRAY, ls="--", lw=1.2)
ax.set_ylim(0, 1); ax.set_xlabel("Age at founding"); ax.set_ylabel("Predicted P(success)")
ax.set_title("Predicted success probability by founder age")
plt.tight_layout(); plt.show()

## 2. P(success | went to an Ivy League school)
Only founders whose education we actually know are compared. Founders with no education info are not 'non-Ivy', they are unknown, so they are excluded.

In [ ]:
ivy_df = df[df["edu_known"] & df["success"].notna()].copy()
ivy_df["ivy_group"] = np.where(ivy_df["ivy"], "Ivy League", "Other schools")

ivy_tbl = rate_table(ivy_df, "ivy_group", order=["Ivy League", "Other schools"])
display(ivy_tbl.round(3))
plot_rates(ivy_tbl, "Success rate: Ivy League vs. other schools", "",
           baseline=ivy_df["success"].mean(), colors=[ORANGE, BLUE])

ct = pd.crosstab(ivy_df["ivy_group"], ivy_df["success"])
odds, p_fisher = stats.fisher_exact(ct.values)
print(ct, f"\nFisher exact test: odds ratio = {odds:.2f}, p = {p_fisher:.3f}")

In [ ]:
ivy_m = ivy_df.dropna(subset=["founding_year_c"]).copy()
ivy_m["ivy_int"] = ivy_m["ivy"].astype(int)

mi1 = fit_glm("success ~ ivy_int", ivy_m)
mi2 = fit_glm("success ~ ivy_int + founding_year_c + C(industry_grp)", ivy_m)
print(mi2.summary())
display(odds_ratios(mi1).loc[["ivy_int"]].rename(index={"ivy_int": "ivy (no controls)"}))
display(odds_ratios(mi2).loc[["ivy_int"]].rename(index={"ivy_int": "ivy (+ founding year, industry)"}))

## 3. Master's vs. bachelor's only (and PhD, dropouts)

In [ ]:
DEG_ORDER  = ["bachelors_only", "masters", "phd", "dropout"]
DEG_LABELS = ["Bachelor's only", "Master's / MBA", "PhD", "Dropout"]
deg_df = df[df["degree_level"].notna() & df["success"].notna()].copy()

deg_tbl = rate_table(deg_df, "degree_level", order=DEG_ORDER).dropna(how="all")
display(deg_tbl.round(3))
labels = [DEG_LABELS[DEG_ORDER.index(i)] for i in deg_tbl.index]
plot_rates(deg_tbl, "Success rate by highest degree", "", baseline=deg_df["success"].mean(),
           colors=[BLUE, ORANGE, "#1baf7a", GRAY][:len(deg_tbl)], labels=labels)

ct = pd.crosstab(deg_df["degree_level"], deg_df["success"])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct.values)
print(ct, f"\nChi-square test across degree levels: chi2 = {chi2:.2f}, dof = {dof}, p = {p_chi:.3f}")

In [ ]:
deg_m = deg_df.dropna(subset=["founding_year_c"]).copy()
md1 = fit_glm("success ~ C(degree_level, Treatment('bachelors_only'))", deg_m)
md2 = fit_glm("success ~ C(degree_level, Treatment('bachelors_only')) + founding_year_c + C(industry_grp)", deg_m)
print(md2.summary())
display(odds_ratios(md1))

## 4. Bonus: success by YC batch
Useful if "batch" in the professor's note meant the YC batch. Read with care: older batches have had more time to resolve, so recent batches look different partly because outcomes are not decided yet.

In [ ]:
b_df = df.dropna(subset=["batch_year", "success"]).copy()
b_df["batch_year"] = b_df["batch_year"].astype(int)

year_tbl = rate_table(b_df, "batch_year")
display(year_tbl.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(year_tbl.index, year_tbl["p_success"],
            yerr=[year_tbl["p_success"] - year_tbl["ci_low"], year_tbl["ci_high"] - year_tbl["p_success"]],
            fmt="o", color=BLUE, ecolor=GRAY, elinewidth=1.2, capsize=3, ms=6, mfc=BLUE, mec="white")
ax.axhline(b_df["success"].mean(), color=GRAY, ls="--", lw=1.2)
ax.set_ylim(0, 1.05); ax.set_xlabel("YC batch year"); ax.set_ylabel("P(success)")
ax.set_title("Success rate by YC batch year (bars show 95% CI)")
plt.tight_layout(); plt.show()

season_tbl = rate_table(b_df[b_df["batch_season"].isin(["W", "S"])], "batch_season", order=["W", "S"])
display(season_tbl.round(3))

## Read before you report any number
- **Censoring.** Companies still `active` are excluded, so the success rate here is among resolved companies only, not all companies.
- **What "success" means.** `acquired` includes small acqui-hires. Try moving `active` into success, or splitting `public` from `acquired`, as a sensitivity check.
- **Small samples.** Age is known for only part of the rows, and the Ivy and master's groups are small. Report the confidence intervals, not just the point estimates.
- **Not independent rows.** Founders at the same company share an outcome, so the models cluster standard errors by company. Descriptive rates above still count each founder as a row.
- **Keyword-based flags.** Ivy and master's come from text matching. Report that you audited them.
- **Associations, not causes.** These are P(success | X) results. To say age causes success you need the DAG, adjustment set and matching from the main design, not just these models.